# MediScan — Day 5: Base Model Setup

## Objective

Set up a pre-trained EfficientNet-B0 model using transfer learning
and replace its classifier head for four-class brain MRI classification.

The base model will initially be frozen so that the classifier head
can be trained first.

In [1]:
# ============================================
# Day 5 - Imports
# ============================================

import torch
import torch.nn as nn
import torchvision

from torchvision import models

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)

PyTorch version: 2.11.0+cu128
Torchvision version: 0.26.0+cu128


In [2]:
# ============================================
# Day 5 - Device Setup
# ============================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not available")

Device: cuda
GPU: Tesla T4


In [3]:
# ============================================
# Day 5 - Load Pre-trained EfficientNet-B0
# ============================================

weights = models.EfficientNet_B0_Weights.DEFAULT

model = models.efficientnet_b0(
    weights=weights
)

print("Pre-trained EfficientNet-B0 loaded successfully!")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 78.5MB/s]


Pre-trained EfficientNet-B0 loaded successfully!


In [4]:
# ============================================
# Inspect EfficientNet-B0 Classifier
# ============================================

print("Original classifier:")
print(model.classifier)

Original classifier:
Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)


In [5]:
# ============================================
# Day 5 - Modify Classifier Head
# ============================================

NUM_CLASSES = 4

in_features = model.classifier[1].in_features

model.classifier[1] = nn.Linear(
    in_features,
    NUM_CLASSES
)

print("Classifier head modified successfully!")
print(model.classifier)

Classifier head modified successfully!
Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=4, bias=True)
)


In [6]:
# ============================================
# Day 5 - Freeze Base Model
# ============================================

for param in model.features.parameters():
    param.requires_grad = False

print("Base model frozen successfully!")

Base model frozen successfully!


In [7]:
# ============================================
# Day 5 - Move Model to Device
# ============================================

model = model.to(device)

print("Model moved to:", device)

Model moved to: cuda


In [8]:
# ============================================
# Day 5 - Trainable Parameter Verification
# ============================================

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

frozen_params = total_params - trainable_params

print("Total parameters     :", total_params)
print("Trainable parameters :", trainable_params)
print("Frozen parameters    :", frozen_params)

Total parameters     : 4012672
Trainable parameters : 5124
Frozen parameters    : 4007548


In [9]:
# ============================================
# Day 5 - Final Model Verification
# ============================================

print("========================================")
print("DAY 5 MODEL VERIFICATION")
print("========================================")

print("Architecture : EfficientNet-B0")
print("Pre-trained  : Yes")
print("Num classes  :", NUM_CLASSES)
print("Device       :", device)

print("\nClassifier:")
print(model.classifier)

print("\nTotal parameters     :", total_params)
print("Trainable parameters :", trainable_params)
print("Frozen parameters    :", frozen_params)

print("\n========================================")
print("DAY 5 BASE MODEL SETUP PASSED")
print("========================================")

DAY 5 MODEL VERIFICATION
Architecture : EfficientNet-B0
Pre-trained  : Yes
Num classes  : 4
Device       : cuda

Classifier:
Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=4, bias=True)
)

Total parameters     : 4012672
Trainable parameters : 5124
Frozen parameters    : 4007548

DAY 5 BASE MODEL SETUP PASSED
